# RFW all-in-one supplementary verification

이 노트북은 RFW 공식 protocol 검증, 선택 FR checkpoint의 origin embedding 추출·재사용, 명시적으로 선택한 LFW/SurvFace frozen codec 적용과 보조 1:1 verification 보고를 한 번에 수행한다. RFW에서 PCA/PQ를 fit하지 않으며 LFW/SurvFace 1:N open-set 결과와 합치지 않는다.

재개 방법: 커널을 재시작하고 처음부터 실행한다. 완료 protocol·origin·evaluation은 manifest와 SHA가 모두 일치할 때만 재사용한다. 실제 장시간 단계는 `EXECUTE=True`, `ACKNOWLEDGE_LOCAL_EXECUTION=True`와 해당 `RUN_*_STAGE=True`가 모두 필요하다.


## 1. 실행 설정

`CODEC_SOURCE_RUN_DIRS`에는 같은 `MODEL_UID`로 새 코드에서 완료되어 `frozen_codec_manifest.json`을 가진 run만 명시한다. 최신 run을 자동 선택하지 않는다. 비어 있으면 origin-only baseline까지 수행한다.


In [ ]:
from __future__ import annotations

MODEL_UID = "edgeface-a348c305af33c223b337"  # 평가할 모델의 고유 식별자
SEED = 42  # 난수 생성 시드 (재현성 보장)
DEVICE = "cuda"  # 임베딩 추출 연산 장치 (cuda 우선 사용)
BATCH_SIZE = 128  # GPU 메모리에 맞춘 배치 크기 (예: GTX 1080 Ti 11GB)
HORIZONTAL_FLIP_TTA = False  # 좌우 반전(Test-Time Augmentation) 적용 여부
BOOTSTRAP_REPEATS = 500  # 신뢰 구간 추정을 위한 부트스트랩 반복 횟수
ARTIFACT_STORAGE_MODE = "results_only"  # 아티팩트 저장 모드 (전체 파일 저장 여부 결정)

EXECUTE = False  # 실제 시간이 오래 걸리는 단계의 실행 여부
ACKNOWLEDGE_LOCAL_EXECUTION = False  # 로컬 환경(장치 제한)에서 실제 실행됨을 확인
RUN_PROTOCOL_STAGE = False  # RFW 공식 평가 프로토콜 생성 단계 실행 여부
RUN_EMBEDDING_STAGE = False  # 원본(origin) 얼굴 임베딩 추출 단계 실행 여부
RUN_EVALUATION_STAGE = False  # 동결된 압축 코덱 성능 평가 단계 실행 여부
WRITE_OUTPUTS = True  # 생성된 프로토콜이나 평가 결과 저장 여부
REUSE_COMPLETED = True  # 기존에 완료된(매니페스트 및 SHA 일치) 결과 재사용 여부
OVERWRITE_PROTOCOL = False  # 기존 RFW 프로토콜 파일 덮어쓰기 여부
ALLOW_ORIGIN_ONLY = True  # 압축 코덱(codec) 없이 원본(origin)만으로 평가 진행 허용 여부


CODEC_SOURCE_RUN_DIRS = ()
# CODEC_SOURCE_RUN_DIRS = (
#     "runs/lfw_YYYYMMDD/<completed-edgeface-run>",
#     "runs/survface_YYYYMMDD/<completed-edgeface-run>",
# )
SELECTED_CODEC_FAMILIES = ("pca", "pq")
SELECTED_CODEC_PROFILES = None

if ARTIFACT_STORAGE_MODE not in {"results_only", "full"}:
    raise ValueError("ARTIFACT_STORAGE_MODE must be results_only or full")
if EXECUTE and ACKNOWLEDGE_LOCAL_EXECUTION is not True:
    raise RuntimeError("Set ACKNOWLEDGE_LOCAL_EXECUTION=True before execution")
if any((RUN_PROTOCOL_STAGE, RUN_EMBEDDING_STAGE, RUN_EVALUATION_STAGE)) and not EXECUTE:
    raise ValueError("RUN_*_STAGE=True requires EXECUTE=True")


## 2. 공통 Python 단계와 경로


In [ ]:
from pathlib import Path
from pprint import pprint
import sys

import pandas as pd
from IPython.display import display

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun project root could not be located")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.datasets import (
    build_rfw_verification_bundle,
    inspect_rfw_aligned_bin_archive,
    inspect_rfw_sources,
    write_rfw_verification_bundle,
)
from research.experiments import (
    evaluate_rfw_frozen_codecs,
    extract_rfw_origin_embeddings,
    frozen_codec_specs_from_completed_run,
    load_rfw_frozen_codec_evaluation,
    load_rfw_origin_embedding_artifact,
    rfw_frozen_codec_evaluation_uid,
)

RFW_ROOT = PROJECT_ROOT / "data/raw/RFW"
JPG_ARCHIVE = RFW_ROOT / "images/test.tar.gz"
ALIGNED_BIN_ARCHIVE = RFW_ROOT / "bin_for_mxnet/RFW_test.tar.gz"
EXPECTED_ALIGNED_SHA256 = "8259e53ab1b542a9747335f34b2ae48e22a675bce472e391d6b94bc900fde572"
PROTOCOL_DIR = PROJECT_ROOT / "data/interim/rfw"
PAIR_PROTOCOL_PATH = PROTOCOL_DIR / "pair_protocol.csv"
MODEL_SPEC_PATH = PROJECT_ROOT / "runs/step2/model_registry" / f"{MODEL_UID}.json"
ARTIFACT_ROOT = PROJECT_ROOT / ("results" if ARTIFACT_STORAGE_MODE == "results_only" else "runs")
ORIGIN_ARTIFACT_DIR = ARTIFACT_ROOT / "rfw_step7/origin_embeddings" / MODEL_UID
EVALUATION_ROOT = ARTIFACT_ROOT / "rfw_step7/frozen_codec_evaluation" / MODEL_UID
pprint({"project_root": str(PROJECT_ROOT), "model_uid": MODEL_UID, "origin_dir": str(ORIGIN_ARTIFACT_DIR)})


## 3. RFW source와 공식 protocol

JPG와 aligned BIN은 동일 test의 대체 표현이므로 합산하지 않는다. protocol stage를 실행하지 않아도 기존 `_SUCCESS`와 24,000-pair CSV는 읽기 전용으로 검증한다.


In [ ]:
source_inventory = inspect_rfw_sources(
    RFW_ROOT, project_root=PROJECT_ROOT, verify_sha256=False
)
if RUN_PROTOCOL_STAGE:
    protocol_bundle = build_rfw_verification_bundle(
        JPG_ARCHIVE, PROJECT_ROOT, strict_official=True
    )
    if WRITE_OUTPUTS:
        write_rfw_verification_bundle(
            protocol_bundle, PROTOCOL_DIR, overwrite=OVERWRITE_PROTOCOL
        )
if not (PROTOCOL_DIR / "_SUCCESS").is_file() or not PAIR_PROTOCOL_PATH.is_file():
    raise FileNotFoundError(
        "RFW protocol artifact is absent. Enable RUN_PROTOCOL_STAGE and restart/run all."
    )
pairs = pd.read_csv(PAIR_PROTOCOL_PATH)
if len(pairs) != 24000:
    raise ValueError(f"official RFW evaluation requires 24,000 pairs, got {len(pairs)}")
protocol_counts = pairs.groupby(["rfw_group", "fold_index", "is_genuine"]).size()
if len(protocol_counts) != 80 or not protocol_counts.eq(300).all():
    raise ValueError("RFW protocol must contain 300 genuine/impostor pairs per group/fold")
{"inventory": source_inventory.summary, "pair_count": len(pairs)}


## 4. Origin embedding 추출 또는 완료 artifact 재사용


In [ ]:
archive_summary = inspect_rfw_aligned_bin_archive(
    ALIGNED_BIN_ARCHIVE,
    expected_sha256=EXPECTED_ALIGNED_SHA256,
    strict_official=True,
)
origin_artifact = None
if RUN_EMBEDDING_STAGE:
    def progress(message, details):
        processed = details.get("processed")
        total = details.get("total")
        if processed in {total} or (processed and total and processed % max(total // 10, 1) == 0):
            print(message, details)
    origin_artifact = extract_rfw_origin_embeddings(
        aligned_bin_archive_path=ALIGNED_BIN_ARCHIVE,
        pairs=pairs,
        model_spec_path=MODEL_SPEC_PATH,
        output_dir=ORIGIN_ARTIFACT_DIR,
        expected_archive_sha256=EXPECTED_ALIGNED_SHA256,
        expected_model_uid=MODEL_UID,
        device=DEVICE,
        batch_size=BATCH_SIZE,
        horizontal_flip_tta=HORIZONTAL_FLIP_TTA,
        strict_official=True,
        reuse_completed=REUSE_COMPLETED,
        progress=progress,
    )
elif (ORIGIN_ARTIFACT_DIR / "_SUCCESS").is_file():
    origin_artifact = load_rfw_origin_embedding_artifact(ORIGIN_ARTIFACT_DIR)
if origin_artifact is None:
    origin_status = {"status": "not_started", "next_action": "enable RUN_EMBEDDING_STAGE"}
else:
    if origin_artifact.manifest["model_uid"] != MODEL_UID:
        raise ValueError("RFW origin artifact model UID mismatch")
    origin_status = {"status": "completed", **origin_artifact.manifest}
origin_status


## 5. 명시적 frozen codec lineage

각 source run은 완료 marker, run/freeze/codec manifest identity, model UID 및 artifact SHA를 통과해야 한다. RFW embedding으로 `fit()`을 호출하는 경로는 없다.


In [ ]:
codec_specs = []
for recorded_run_dir in CODEC_SOURCE_RUN_DIRS:
    source_run_dir = Path(recorded_run_dir)
    if not source_run_dir.is_absolute():
        source_run_dir = PROJECT_ROOT / source_run_dir
    codec_specs.extend(
        frozen_codec_specs_from_completed_run(
            source_run_dir,
            expected_model_uid=MODEL_UID,
            families=SELECTED_CODEC_FAMILIES,
            profile_names=SELECTED_CODEC_PROFILES,
        )
    )
codec_specs = tuple(codec_specs)
if not codec_specs and not ALLOW_ORIGIN_ONLY:
    raise RuntimeError("No frozen codecs selected and origin-only is disabled")
[
    {"profile": spec.profile_name, "family": spec.family, "source": spec.fit_source_run_id}
    for spec in codec_specs
] or [{"status": "origin_only", "compression_claim": False}]


## 6. 10-fold 평가와 compact 결과


In [ ]:
evaluation = None
evaluation_uid = None
evaluation_dir = None
if origin_artifact is not None:
    evaluation_uid = rfw_frozen_codec_evaluation_uid(
        origin_artifact_dir=ORIGIN_ARTIFACT_DIR,
        codec_specs=codec_specs,
        strict_official=True,
        bootstrap_seed=SEED,
        bootstrap_repeats=BOOTSTRAP_REPEATS,
    )
    evaluation_dir = EVALUATION_ROOT / evaluation_uid
    if RUN_EVALUATION_STAGE:
        if not WRITE_OUTPUTS:
            raise ValueError("RFW evaluator produces immutable outputs; WRITE_OUTPUTS=True is required")
        evaluation = evaluate_rfw_frozen_codecs(
            origin_artifact_dir=ORIGIN_ARTIFACT_DIR,
            codec_specs=codec_specs,
            output_dir=evaluation_dir,
            strict_official=True,
            bootstrap_seed=SEED,
            bootstrap_repeats=BOOTSTRAP_REPEATS,
            reuse_completed=REUSE_COMPLETED,
        )
    elif (evaluation_dir / "_SUCCESS").is_file():
        evaluation = load_rfw_frozen_codec_evaluation(evaluation_dir)
evaluation_status = {
    "status": "completed" if evaluation is not None else "not_started",
    "evaluation_uid": evaluation_uid,
    "evaluation_dir": str(evaluation_dir) if evaluation_dir else None,
    "codec_count": len(codec_specs),
}
evaluation_status


## 7. 결과 확인


In [ ]:
if evaluation is not None:
    display(evaluation.profile_summary)
    display(evaluation.group_summary)
    pprint(evaluation.manifest, sort_dicts=False)
else:
    print("No evaluation loaded. Review evaluation_status and enable the required stage.")


## 해석 및 완료 기준

- 완료 조건은 protocol `_SUCCESS`, origin manifest/SHA, evaluation manifest/SHA 및 24,000 pair/48,000 occurrence 일치다.
- `codec_count=0`은 origin-only baseline이며 압축 일반화 결과가 아니다.
- RFW는 supplementary 1:1 verification이다. DIR@FPIR, open-set rank 또는 pgvector 1:N latency를 생성하지 않는다.
- EdgeFace–RFW training identity overlap은 미확인이므로 strict unseen-identity 일반화로 주장하지 않는다.
- BalancedFace는 현재 범위에서 유예한다.
